In [1]:
#from torch import nn
import pandas as pd
import numpy as np
from pathlib import Path
import torch
from nfl.lib import enums

from nfl.data_management.DataManager import DataManager
from nfl.NeuralNetwork.EPA_Predictor import EPAPredictor
from nfl.NeuralNetwork.NNSolver import Solver
from nfl.lib.utils import create_train_test_split
from torch.utils.tensorboard import SummaryWriter
from nfl.NeuralNetwork.ParameterSelector import GridSearch, GeneticSearch

In [2]:
SCENARIO_COLS = [
    "yardline_100",
    "game_seconds_remaining",
    "has_turf",
    "temp",
    "wind",
    "has_roof",
    "ydstogo",
    "goal_to_go",
    "score_differential",
    "down",
    "div_game",
    "day_of_season",
    "series"
]

PLAY_COLS = [
    "play_type",
    "pass_location",
#    "pass_length",
#    "run_location",
    "run_gap",
    "shotgun",
    "no_huddle",
    "qb_kneel",
    "qb_spike",
    "qb_scramble",
    "air_yards"
]

RESULT_COLS = [
    "epa",
    "wpa",
    "success",
    "result",
    "series_success",
    "tackle_for_loss",
    "saftey",
    "yards_gained",
    "touchdown",
    "fumble",
    "complete_pass",
    "rushing_yards",
    "fumble_lost",
    "interception",
    "sack",
    "penalty_yards",
]

EXCLUDED_PLAY_TYPES = {
    enums.PlayType.KICK,
    enums.PlayType.EXTRA_POINT,
    enums.PlayType.NO_PLAY,
    enums.PlayType.GAME_START,
}

In [3]:
data = DataManager.get_data(path_to_json = (Path.cwd() / "nfl/data").resolve())
data = DataManager.clean_data(data, excluded_play_types=EXCLUDED_PLAY_TYPES, target_col="epa")
data, feature_cols = DataManager.prepare_features(data, scenario_columns=SCENARIO_COLS, play_columns=PLAY_COLS)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Cuda device count: ", torch.cuda.device_count())
print(f"Using {device} device")

Cuda device count:  1
Using cuda device


In [5]:
pd.set_option('display.max_columns', None)
data.head(5)

,yardline_100,game_seconds_remaining,quarter_end,drive,sp,qtr,down,goal_to_go,ydstogo,ydsnet,yards_gained,shotgun,no_huddle,qb_dropback,qb_kneel,qb_spike,qb_scramble,pass_length,air_yards,yards_after_catch,run_gap,field_goal_result,extra_point_result,score_differential,score_differential_post,ep,epa,wp,home_wp,wpa,home_wp_post,air_wpa,yac_wpa,comp_air_wpa,comp_yac_wpa,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,fourth_down_converted,fourth_down_failed,incomplete_pass,touchback,interception,fumble_forced,fumble_not_forced,fumble_out_of_bounds,solo_tackle,safety,penalty,tackled_for_loss,fumble_lost,qb_hit,rush_attempt,pass_attempt,sack,touchdown,pass_touchdown,rush_touchdown,return_touchdown,extra_point_attempt,field_goal_attempt,fumble,complete_pass,assist_tackle,passing_yards,receiving_yards,rushing_yards,tackle_with_assist,fumble_recovery_1_yards,fumble_recovery_2_yards,return_yards,penalty_yards,replay_or_challenge,replay_or_challenge_result,penalty_type,season,cp,cpoe,series,series_success,series_result,order_sequence,play_clock,play_type_nfl,st_play_type,end_yard_line,drive_play_count,drive_first_downs,drive_ended_with_score,drive_quarter_start,drive_quarter_end,drive_yards_penalized,drive_start_transition,drive_end_transition,drive_game_clock_start,drive_game_clock_end,drive_start_yard_line,drive_end_yard_line,result,div_game,temp,wind,aborted_play,success,pass,rush,first_down,special,play,out_of_bounds,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe,day_of_season,has_roof,has_turf,play_type_field_goal,play_type_pass,play_type_punt,play_type_qb_kneel,play_type_qb_spike,play_type_run
2,80.0,3600.0,0.0,1.0,0.0,1,1.0,0.0,10.0,18.0,3.0,0.0,0.0,1.0,0.0,0.0,0.0,short,3.0,0.0,-5.0,0.0,0.0,0.0,0.0,0.239785,-0.337139,0.422024,0.577976,-0.001425,0.579401,-0.001425,0.000000,-0.001425,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015,0.765811,23.418927,1.0,1.0,First down,51.0,12.0,PASS,0.0,23.0,6.0,1.0,0.0,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,0.0,88.0,13.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,4.699278,3.0,0.678964,0.225919,0.456481,54.351911,43.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,77.0,3573.0,0.0,1.0,0.0,1,2.0,0.0,7.0,18.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-0.097354,-0.262481,0.420599,0.579401,-0.017304,0.596705,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015,0.000000,0.000000,1.0,1.0,First down,75.0,18.0,RUSH,0.0,25.0,6.0,1.0,0.0,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,0.0,88.0,13.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.000000,0.0,0.000000,0.000000,0.545905,-54.590458,43.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,75.0,3532.0,0.0,1.0,0.0,1,3.0,0.0,5.0,18.0,10.0,1.0,0.0,1.0,0.0,0.0,0.0,short,4.0,6.0,-5.0,0.0,0.0,0.0,0.0,-0.359835,1.661242,0.403295,0.596705,0.045358,0.551347,0.000000,0.045358,0.000000,0.045358,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,10.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015,0.510176,48.982388,1.0,1.0,First down,96.0,5.0,PASS,0.0,35.0,6.0,1.0,0.0,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,0.0,88.0,13.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,4.662650,2.0,0.712350,0.712350,0.968533,3.146732,43.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5,65.0,3494.0,0.0,1.0,0.0,1,1.0,0.0,10.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,1.301407,-0.518931,0.448653,0.551347,-0.018066,0.569413,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015,0.000000,0.000000,2.0,0.0,Punt,120.0,13.0,RUSH,0.0,35.0

In [6]:
split = create_train_test_split(df=data, feature_cols=feature_cols, target_col="epa", train_frac = 0.8, device=device)
X_train, y_train, X_test, y_test, scaler = split

In [ ]:
EPA_INPUT_SIZE = len(feature_cols)
NUM_ITER = 7
NUM_EPOCH = 50
EPS = 1e-8
BETAS = (0.9, 0.999)

MIN_HIDDEN_LAYERS, MAX_HIDDEN_LAYERS = 1, 20
MIN_HIDDEN_SIZE, MAX_HIDDEN_SIZE = 16, 256
MIN_LEARNING_RATE, MAX_LEARNING_RATE = 1e-5, 1e-2
MIN_BATCH_SIZE, MAX_BATCH_SIZE = 64, 512
MIN_WEIGHT_DECAY, MAX_WEIGHT_DECAY = 1e-5, 1e-2

BATCH_SIZE = np.linspace(MIN_BATCH_SIZE, MAX_BATCH_SIZE, num=NUM_ITER).astype(int).tolist()
LEARNING_RATE = np.geomspace(MIN_LEARNING_RATE, MAX_LEARNING_RATE, num=NUM_ITER).tolist()
NUM_HIDDEN_LAYERS = np.linspace(MIN_HIDDEN_LAYERS, MAX_HIDDEN_LAYERS, num=NUM_ITER).astype(int).tolist()
HIDDEN_SIZE = np.linspace(MIN_HIDDEN_SIZE, MAX_HIDDEN_SIZE, num=NUM_ITER).astype(int).tolist()
WEIGHT_DECAY = np.geomspace(MIN_WEIGHT_DECAY, MAX_WEIGHT_DECAY, num=NUM_ITER).tolist()

param_grid = {
    "batch_size": BATCH_SIZE,
    "lr": LEARNING_RATE,
    "num_hidden_layers": NUM_HIDDEN_LAYERS,
    "hidden_size": HIDDEN_SIZE,
    "weight_decay": WEIGHT_DECAY,
}

In [ ]:
pop_size = len(param_grid)*3
top_k = pop_size//4
num_generations = 4*len(param_grid)

ga_search = GeneticSearch(
    param_grid=param_grid,
    solver_cls=Solver,
    model_cls=EPAPredictor,
    criterion=torch.nn.MSELoss().to(device),
    device=device,
    NUM_EPOCH=NUM_EPOCH,
    input_size=EPA_INPUT_SIZE,
    pop_size=pop_size,
    generations=num_generations,
    mutation_rate=0.25,
    top_k=top_k,
    log_dir="runs/ga_sweep"
)

results_df, best_config = ga_search.run(X_train, y_train, X_test, y_test, metric="best_val_loss")

print("\nBest Model Found:")
print(best_config)

# 2. Save results to CSV for inspection
results_df.to_csv("ga_sweep_results.csv", index=False)

--- Starting Genetic Search (5 generations, pop size 12) ---

=== Generation 1/5 ===
Epoch   0/100 | Train Loss: 1.8589 | Valid Loss: 1.8344
Epoch  20/100 | Train Loss: 1.8589 | Valid Loss: 1.8344
Epoch  40/100 | Train Loss: 1.8589 | Valid Loss: 1.8344
Epoch  60/100 | Train Loss: 1.8589 | Valid Loss: 1.8344
Epoch  80/100 | Train Loss: 1.8589 | Valid Loss: 1.8344
Epoch  99/100 | Train Loss: 1.8589 | Valid Loss: 1.8344
Ind [1/12]: best_val_loss = 1.8344
